# Strained CALPHAD model for wurtzite Al$_{1-x}$Sc$_x$N

Phase separation of wurtzite Al$_{1-x}$Sc$_x$N as bulk and as a film on a substrate: binodals, spinodals, critical
temperatures and the decomposition driving force, for incoherent and coherent domains.

Every literature number here was extracted and validated in the companion notebook
`Hirata2020_AlScN_CALPHAD.ipynb`; each input cell names the section it comes from. Energies are J per mol of formula
unit internally and meV per formula unit (f.u. = Al$_{1-x}$Sc$_x$N) in tables and plots.

Navigation: section 2 holds everything adjustable, section 4 the physics, section 6 the results. Sections 1, 3 and 5
are machinery and can stay collapsed.

Sources: K. Hirata *et al.*, *J. Eur. Ceram. Soc.* **40**, 5410 (2020),
[10.1016/j.jeurceramsoc.2020.06.047](https://doi.org/10.1016/j.jeurceramsoc.2020.06.047) (free energy, lattice
parameters, isotropic elastic moduli); S. Zhang *et al.*, *J. Appl. Phys.* **114**, 243516 (2013),
[10.1063/1.4848036](https://doi.org/10.1063/1.4848036) (hexagonal elastic constants, critical thickness); D. F. Urban
*et al.*, *Phys. Rev. B* **103**, 115204 (2021), [10.1103/PhysRevB.103.115204](https://doi.org/10.1103/PhysRevB.103.115204)
(hexagonal elastic constants, h-ScN); D. Holec, PhD thesis, Cambridge (2008) (dislocation cores).

# 1. Smooth-function machinery

Shared by the interpolated inputs of section 2. Nothing physical here.

In [ ]:
import concurrent.futures
import dataclasses
import functools
import math
import multiprocessing
import operator

import numpy as np
from scipy.optimize import brentq, minimize, minimize_scalar
from scipy.special import expit, xlogy

try:
    import matplotlib.pyplot as plt
    HAVE_MATPLOTLIB = True
except ImportError:
    HAVE_MATPLOTLIB = False


def softplus(value, width):
    """Smooth max(value, 0) with the given width; accepts arrays."""
    return width * np.logaddexp(0.0, np.asarray(value, dtype=float) / width)


def smoothed_branch_fit(composition, parameters, width):
    """Two-branch fit [p0, p1, p2, p4] with its sqrt and kink at x0 replaced by softplus forms."""
    x = np.asarray(composition, dtype=float)
    above = softplus(x - TRANSITION_COMPOSITION, width)
    return (parameters[0] + parameters[1] * (x - above) + parameters[3] * above
            + parameters[2] * np.sqrt(softplus(TRANSITION_COMPOSITION - x, 0.5 * width)))


def joined_branches(composition, polynomials, width):
    """Buckled and flat polynomial fits blended by a logistic weight centred on x0."""
    x = np.asarray(composition, dtype=float)
    weight = expit((x - TRANSITION_COMPOSITION) / width)
    return (1.0 - weight) * np.polyval(polynomials[0], x) + weight * np.polyval(polynomials[1], x)

# 2. Inputs

Everything adjustable lives here.

## 2.1 Literature data

### 2.1.1 Hirata et al. (2020) — assessed free energy

#### 2.1.1.1 Tabulated

In [ ]:
# Wurtzite (Al,Sc)N Redlich-Kister parameters, J per mol of formula unit. Table 2; the sign of L2 is lost from the PDF
# text and fixed by Fig. 14 - Hirata notebook sections 5 and 6a.
REDLICH_KISTER_WURTZITE = (97970.4838186, 434.322298892, -17960.9058372)    # L0, L1, L2

### 2.1.2 Hirata et al. (2020) — lattice parameters

#### 2.1.2.1 Tabulated

In [ ]:
# Continuity-constrained two-branch fit to the Fig. 11 data, Angstrom - Hirata notebook section 15 (printed in 16c).
# Parameters [p0, p1, p2, p4]: x < x0: p0 + p1 x + p2 sqrt(x0 - x);  x > x0: p0 + p1 x0 + p4 (x - x0).
TRANSITION_COMPOSITION = 0.5555                                      # x0, buckled -> flat
LATTICE_A_FIT = (3.398373741033616, 0.17544525954074075, -0.38563833826503907, 0.47273298890089305)
LATTICE_C_FIT = (3.5607476553295663, 1.5834721310585416, 1.9040612417880973, 0.09698571729425413)

#### 2.1.2.2 Interpolants

In [ ]:
def lattice_parameters(composition):
    """Stress-free (a, c) in Angstrom of a homogeneous alloy, transition smoothed over lattice_transition_width."""
    return (smoothed_branch_fit(composition, LATTICE_A_FIT, lattice_transition_width),
            smoothed_branch_fit(composition, LATTICE_C_FIT, lattice_transition_width))

### 2.1.3 Hirata et al. (2020) — elastic moduli

#### 2.1.3.1 Tabulated

In [ ]:
# Isotropic (Voigt-Reuss-Hill) moduli of Fig. 10: quadratic fit on the buckled branch (x <= 0.5), linear on the flat
# branch (x >= 0.75), highest power first - Hirata notebook section 16a. E within 1.5 %, nu within 0.0016.
YOUNGS_MODULUS_BRANCH_FITS = ((135.04, -409.52, 342.41), (18.32, 159.37333333333305))         # GPa
POISSON_RATIO_BRANCH_FITS = ((-0.0777142857142858, 0.21805714285714273, 0.24555142857142848),
                             (-0.011599999999999593, 0.3536166666666662))

#### 2.1.3.2 Interpolants

In [ ]:
def youngs_modulus(composition):
    """Isotropic Young's modulus, Pa."""
    return joined_branches(composition, YOUNGS_MODULUS_BRANCH_FITS, lattice_transition_width) * 1e9


def poisson_ratio(composition):
    return joined_branches(composition, POISSON_RATIO_BRANCH_FITS, lattice_transition_width)

### 2.1.4 Zhang et al. (2013) and Urban et al. (2021) — hexagonal elastic constants

#### 2.1.4.1 Tabulated

In [ ]:
# Zhang appendix fits (slope, intercept), GPa, 0 <= x <= 0.375; signs reconstructed - Hirata notebook section 19b.
ZHANG_ELASTIC_FITS = {"C11": (-262.692, 389.466), "C12": (24.774, 135.847), "C13": (62.072, 108.197),
                      "C33": (-459.210, 364.648), "C44": (-50.405, 115.971)}
# Urban Eqs. (8)-(12), (value at x = 0, b, c) of value (1 + b x + c x^2), GPa, 0 <= x <= 0.5 - Hirata notebook 21b.
URBAN_ELASTIC_FITS = {"C11": (374.1, -0.882, 0.602), "C12": (128.6, 0.400, -0.082), "C13": (100.3, 0.793, -0.481),
                      "C33": (351.7, -1.160, -0.256), "C44": (111.6, -0.848, 1.369)}
# Urban Table II, hexagonal ScN (wurtzite with u = 1/2), GPa - Hirata notebook 21a.
URBAN_HEXAGONAL_SCANDIUM_NITRIDE = {"C11": 218.0, "C12": 160.0, "C13": 89.0, "C33": 346.0, "C44": 132.0}
ELASTIC_CONSTANT_NAMES = ("C11", "C12", "C13", "C33", "C44")

#### 2.1.4.2 Interpolants

Each fit is extrapolated up to $x_0$, then joined linearly to Urban's h-ScN at $x = 1$. The two branches are blended
like $E$, $\nu$, so a constant has no kink at $x_0$. Validated (mechanical stability, plots) in Hirata notebook section 22.

In [ ]:
def zhang_polynomials():
    """{name: numpy.polyval coefficients} of Zhang's fits, GPa."""
    return {name: np.array(ZHANG_ELASTIC_FITS[name]) for name in ELASTIC_CONSTANT_NAMES}


def urban_polynomials():
    """{name: numpy.polyval coefficients} of Urban's Eqs. (8)-(12), GPa."""
    return {name: URBAN_ELASTIC_FITS[name][0] * np.array([URBAN_ELASTIC_FITS[name][2], URBAN_ELASTIC_FITS[name][1], 1.0])
            for name in ELASTIC_CONSTANT_NAMES}


def joined_to_hexagonal_scandium_nitride(low_polynomials):
    """{name: (low fit, line from the fit at x0 to h-ScN at x = 1)}, for joined_branches."""
    branches = {}
    for name, low in low_polynomials.items():
        at_transition = np.polyval(low, TRANSITION_COMPOSITION)
        slope = (URBAN_HEXAGONAL_SCANDIUM_NITRIDE[name] - at_transition) / (1.0 - TRANSITION_COMPOSITION)
        branches[name] = (low, np.array([slope, URBAN_HEXAGONAL_SCANDIUM_NITRIDE[name] - slope]))
    return branches


zhang_elastic_branches = joined_to_hexagonal_scandium_nitride(zhang_polynomials())
urban_elastic_branches = joined_to_hexagonal_scandium_nitride(urban_polynomials())

### 2.1.5 Hirata et al. (2020) — misfit relaxation

In [ ]:
def strain_relaxation_factor(film_thickness_nm, critical_thickness_nm):
    """Fraction of the coherent strain energy kept above h_c: (h_c/h)(1 + ln(h/h_c)). Hirata Eq. (21), section 7."""
    if film_thickness_nm <= critical_thickness_nm:
        return 1.0
    ratio = critical_thickness_nm / film_thickness_nm
    return ratio * (1.0 + math.log(film_thickness_nm / critical_thickness_nm))

### 2.1.6 Substrates

In [ ]:
# In-plane lattice parameter, Angstrom. Experimental; assumed on the same scale as Hirata's a(x), whose a(0) is within
# 0.03 % of experimental AlN. AlN itself is taken from the model's own a(0) in 2.2.5.
SUBSTRATE_IN_PLANE_LATTICE = {"GaN": 3.189}

## 2.2 Model inputs

Not from any paper — chosen here, and meant to be edited. Every assumed value is marked.

### 2.2.1 Free-energy corrections

In [ ]:
# Optional corrections to Hirata's G, off by default: add "short_range_order" and/or "excess_vibrational_entropy".
chemical_corrections = ()
# Short-range order: second-order estimate on a cation sublattice of coordination z. A high-temperature expansion,
# not valid far below T_c.
cation_coordination_number = 12
# Excess vibrational entropy of mixing at x = 0.5, k_B per atom. ASSUMED: typical range 0.1-0.2.
excess_vibrational_entropy_kb_per_atom_at_half = 0.15

### 2.2.2 Assumed shape parameters

In [ ]:
# Width over which the buckled -> flat transition at x0 is smeared in a(x), c(x), E(x), nu(x). ASSUMED; the
# sqrt fit itself has an infinite slope at x0. Controls the strength of every effect near x0.
lattice_transition_width = 0.02
# Smooth floor of the critical thickness at the dislocation core radius, nm. ASSUMED.
core_radius_floor_width_nm = 0.05

### 2.2.3 Elastic data

| option | $x \le x_0$ | $x_0 \le x \le 1$ |
|---|---|---|
| `"urban"` | Urban's hexagonal $C_{ij}$, extrapolated from $x = 0.5$ | linear to Urban's h-ScN |
| `"zhang"` | Zhang's hexagonal $C_{ij}$, extrapolated from $x = 0.375$ | linear to Urban's h-ScN |
| `"hirata_isotropic"` | Hirata's isotropic $E$, $\nu$ (the earlier default) | Hirata's flat branch |

**ASSUMED:** the extrapolations to $x_0$ and the linear join to h-ScN, both of which make $C_{33}$ dip to ~100 GPa at
$x_0$. The dislocation line energy uses the Voigt–Reuss–Hill isotropic average of the chosen constants.

In [ ]:
chosen_elastic_data = "urban"      # "urban", "zhang" or "hirata_isotropic"; every result below uses it

### 2.2.4 Dislocation core parameters

$a+c$-type core energy $E_c(x)$ (eV/Å) and radius $r_c(x)$ (nm) for the critical thickness. **All DUMMIES**: atomistic
values exist only for GaN and for $a$-type cores; scaling GaN $a+c$ by the AlN/GaN $a$-type ratio gives 3.36 eV/Å and
0.92 nm for AlN (Hirata notebook section 20). ScN: no data.

In [ ]:
def dummy_core_energy_high_constant(composition):
    """eV/Angstrom, the AlN estimate at every x. DUMMY."""
    return 3.3 + 0.0 * np.asarray(composition, dtype=float)


def dummy_core_energy_low_constant(composition):
    """eV/Angstrom, 3.1 at every x. DUMMY."""
    return 3.1 + 0.0 * np.asarray(composition, dtype=float)


def dummy_core_energy_high_slope(composition):
    """eV/Angstrom, linear from 3.3 (AlN) to 3.0 (ScN). DUMMY."""
    return 3.3 - 0.3 * np.asarray(composition, dtype=float)


def dummy_core_radius(composition):
    """nm, GaN a+c scaled by the AlN/GaN a-type ratio. DUMMY."""
    return 0.72 * 0.83 / 0.65 + 0.0 * np.asarray(composition, dtype=float)

### 2.2.5 Film and growth conditions

In [ ]:
growth_temperatures = (573.15, 673.15, 723.15)          # K: 300, 400, 450 C
growth_temperature = 673.15
real_film_thickness_nm = 1000.0
# In-plane lattice of each substrate, Angstrom; AlN = the model's own a(0), so the misfit vanishes at x = 0.
substrate_in_plane_lattice = {"AlN": float(lattice_parameters(0.0)[0]), **SUBSTRATE_IN_PLANE_LATTICE}
aluminium_nitride = substrate_in_plane_lattice["AlN"]
chosen_core_energy = dummy_core_energy_high_slope        # used by every thickness-dependent result
chosen_core_radius = dummy_core_radius

### 2.2.6 Numerical settings

In [ ]:
interface_angles = np.linspace(0.0, 0.5 * math.pi, 19)   # coherent laminate: 0 = basal, 90 deg = prism
partner_grid_points = 48            # starting-point grid per axis, spaced finely near the average composition
refinement_starts = 3               # best grid pairs refined, and as many well-separated ones
refinement_method = "L-BFGS-B"      # gradient-based; "Nelder-Mead" is the derivative-free backup
small_split = 2e-3                  # extra starting point x -+ small_split
separation = 0.05                   # minimum x_C - x_B of the well-separated starting pairs
composition_margin = 1e-7
fluctuation_amplitude = 1e-3        # split used for the spinodal curvature
decomposition_threshold = 1e-3      # J/mol-f.u.: binodal = where the driving force first exceeds this
binodal_temperatures = np.arange(300.0, 7400.0, 250.0)
binodal_scan = np.linspace(0.01, 0.99, 33)
spinodal_temperatures = np.arange(300.0, 7400.0, 100.0)
spinodal_compositions = np.linspace(0.005, 0.995, 199)
critical_compositions = np.linspace(0.01, 0.99, 197)
driving_force_compositions = np.linspace(0.02, 0.98, 49)
worker_processes = 8                # parallel processes for section 6 (independent temperatures); 1 = serial

# 3. Physical constants

In [ ]:
GAS_CONSTANT = 8.314462618                 # J/(mol K)
JOULE_PER_ELECTRONVOLT = 96485.33212       # J/mol per eV per particle
MILLIELECTRONVOLT_PER_JOULE_PER_MOLE = 1000.0 / JOULE_PER_ELECTRONVOLT
AVOGADRO_CONSTANT = 6.02214076e23          # 1/mol
ELECTRONVOLT_PER_ANGSTROM_IN_JOULE_PER_METRE = 1.602176634e-19 / 1e-10
CUBIC_METRE_PER_CUBIC_ANGSTROM = 1e-30
METRE_PER_ANGSTROM = 1e-10
METRE_PER_NANOMETRE = 1e-9
FORMULA_UNITS_PER_WURTZITE_CELL = 2
ATOMS_PER_FORMULA_UNIT = 2

# 4. Model

4.1 holds the ingredients every case uses, 4.2 the elastic energy of a decomposed film for each geometry, 4.3 the
decomposition and stability tests at one composition, 4.4 a case (one physical situation) as a single object, 4.5 the
phase boundaries, driving-force maps and critical temperatures computed from a case.

## 4.1 Shared ingredients

### 4.1.1 Chemical free energy

$G(x,T) = x(1-x)\sum_n L^{(n)}(1-2x)^n - T S_\mathrm{conf}(x)$ relative to the AlN–ScN wurtzite endmembers, ideal mixing
on the cation sublattice, plus the optional corrections of 2.2.1. One function spans both the buckled and flat
branches, as in Hirata.

In [ ]:
def excess_gibbs_energy(composition):
    """Redlich-Kister excess Gibbs energy of wurtzite, J/mol-f.u.; accepts arrays."""
    x = np.asarray(composition, dtype=float)
    return x * (1.0 - x) * sum(coefficient * (1.0 - 2.0 * x) ** order
                               for order, coefficient in enumerate(REDLICH_KISTER_WURTZITE))


def configurational_entropy(composition):
    """Ideal cation-sublattice mixing entropy, J/(mol-f.u. K); accepts arrays."""
    x = np.asarray(composition, dtype=float)
    return -GAS_CONSTANT * (xlogy(x, x) + xlogy(1.0 - x, 1.0 - x))


def short_range_order_correction(composition, temperature):
    """Second-order (Onsager/TAP) SRO free energy, J/mol-f.u."""
    x = np.asarray(composition, dtype=float)
    interaction = REDLICH_KISTER_WURTZITE[0]
    return -interaction ** 2 * x ** 2 * (1.0 - x) ** 2 / (
        cation_coordination_number * GAS_CONSTANT * temperature)


def excess_vibrational_entropy(composition):
    """Guessed sigma*x(1-x) excess vibrational entropy of mixing, J/(mol-f.u. K)."""
    x = np.asarray(composition, dtype=float)
    per_formula_unit_at_half = (excess_vibrational_entropy_kb_per_atom_at_half * GAS_CONSTANT
                                * ATOMS_PER_FORMULA_UNIT)
    return 4.0 * per_formula_unit_at_half * x * (1.0 - x)


def chemical_gibbs_energy_of_mixing(composition, temperature, corrections=()):
    """Wurtzite G_mix(x, T), J/mol-f.u.; corrections: 'short_range_order', 'excess_vibrational_entropy'."""
    x = np.asarray(composition, dtype=float)
    energy = (excess_gibbs_energy(x)
              - temperature * configurational_entropy(x))
    if "short_range_order" in corrections:
        energy = energy + short_range_order_correction(x, temperature)
    if "excess_vibrational_entropy" in corrections:
        energy = energy - temperature * excess_vibrational_entropy(x)
    return energy


wurtzite_gibbs_energy = functools.partial(chemical_gibbs_energy_of_mixing, corrections=chemical_corrections)

### 4.1.2 Elastic properties of a homogeneous film

Every elastic data option is a set of hexagonal constants $C_{11}, C_{12}, C_{13}, C_{33}, C_{44}(x)$, $c$ along $z$.
Hirata's isotropic $E$, $\nu$ enter as $C_{11} = C_{33} = \lambda + 2\mu$, $C_{12} = C_{13} = \lambda$,
$C_{44} = \mu$. Each geometry needs one of the following:

- A (0001) film clamped in-plane keeps its $c$ free. Its biaxial modulus is $M = C_{11}+C_{12}-2C_{13}^2/C_{33}$ and
  its out-of-plane strain is $-2C_{13}/C_{33}$ times the in-plane one.
- A coherent interface whose normal is at $\theta$ from $c$ has plane stress in the plane spanned by $a$ and the
  direction at $\theta$ from the basal plane. Its reduced stiffness is the inverse of the in-plane compliance:
  $S'_{11} = S_{11}$, $S'_{12} = S_{12}\cos^2\theta + S_{13}\sin^2\theta$,
  $S'_{22} = S_{11}\cos^4\theta + S_{33}\sin^4\theta + (2S_{13}+S_{44})\sin^2\theta\cos^2\theta$.
  For isotropic constants it is independent of $\theta$.
- The dislocation line energy is taken as isotropic, with the Voigt–Reuss–Hill $E$, $\nu$ of the constants (exact for
  the isotropic option).

In [ ]:
def molar_volume(composition):
    """m^3 per mol of formula unit."""
    a, c = lattice_parameters(composition)
    cell_volume = np.sqrt(3.0) / 2.0 * a * a * c * CUBIC_METRE_PER_CUBIC_ANGSTROM
    return cell_volume * AVOGADRO_CONSTANT / FORMULA_UNITS_PER_WURTZITE_CELL


def hexagonal_compliances(c11, c12, c13, c33, c44):
    """(S11, S12, S13, S33, S44) of a hexagonal crystal, inverse units of the constants."""
    determinant = c33 * (c11 + c12) - 2.0 * c13 ** 2
    total, difference = c33 / determinant, 1.0 / (c11 - c12)
    return 0.5 * (total + difference), 0.5 * (total - difference), -c13 / determinant, (c11 + c12) / determinant, 1.0 / c44


def hill_moduli(c11, c12, c13, c33, c44):
    """Voigt-Reuss-Hill (E, nu) of a hexagonal crystal; E in the units of the constants."""
    c66, determinant = 0.5 * (c11 - c12), c33 * (c11 + c12) - 2.0 * c13 ** 2
    bulk_voigt = (2.0 * (c11 + c12) + c33 + 4.0 * c13) / 9.0
    shear_voigt = (c11 + c12 + 2.0 * c33 - 4.0 * c13 + 12.0 * c44 + 12.0 * c66) / 30.0
    bulk_reuss = determinant / (c11 + c12 + 2.0 * c33 - 4.0 * c13)
    shear_reuss = 2.5 * determinant * c44 * c66 / (3.0 * bulk_voigt * c44 * c66 + determinant * (c44 + c66))
    bulk, shear = 0.5 * (bulk_voigt + bulk_reuss), 0.5 * (shear_voigt + shear_reuss)
    return 9.0 * bulk * shear / (3.0 * bulk + shear), (3.0 * bulk - 2.0 * shear) / (2.0 * (3.0 * bulk + shear))


class ElasticConstants:
    """Hexagonal elastic constants of a homogeneous alloy, c along z; methods accept arrays of compositions."""

    def voigt_constants(self, compositions):
        """(C11, C12, C13, C33, C44) in Pa."""
        raise NotImplementedError

    def biaxial_modulus(self, compositions):
        """(0001) biaxial modulus M = C11 + C12 - 2 C13^2 / C33, Pa."""
        c11, c12, c13, c33, _ = self.voigt_constants(compositions)
        return c11 + c12 - 2.0 * c13 ** 2 / c33

    def out_of_plane_coefficient(self, compositions):
        """-eps_zz / eps_xx of a (0001) film under biaxial in-plane strain: 2 C13 / C33."""
        _, _, c13, c33, _ = self.voigt_constants(compositions)
        return 2.0 * c13 / c33

    def plane_stress_stiffness(self, compositions, angles):
        """Stiffness (Pa) in the plane of (a, direction at `angles` from the basal plane); (..., angles, 2, 2)."""
        s11, s12, s13, s33, s44 = (value[..., None] for value in
                                   hexagonal_compliances(*self.voigt_constants(compositions)))
        cosine, sine = np.cos(angles) ** 2, np.sin(angles) ** 2
        cross = s12 * cosine + s13 * sine
        tilted = s11 * cosine ** 2 + s33 * sine ** 2 + (2.0 * s13 + s44) * sine * cosine
        compliance = np.stack([np.stack([np.broadcast_to(s11, tilted.shape), cross], -1),
                               np.stack([cross, tilted], -1)], -2)
        return np.linalg.inv(compliance)

    def isotropic_moduli(self, compositions):
        """Voigt-Reuss-Hill (E in Pa, nu)."""
        return hill_moduli(*self.voigt_constants(compositions))


class IsotropicElasticConstants(ElasticConstants):
    """Hirata's isotropic (VRH) E(x), nu(x) of 2.1.3."""

    def voigt_constants(self, compositions):
        youngs, poisson = youngs_modulus(compositions), poisson_ratio(compositions)
        lame = youngs * poisson / ((1.0 + poisson) * (1.0 - 2.0 * poisson))
        shear = youngs / (2.0 * (1.0 + poisson))
        return lame + 2.0 * shear, lame, lame, lame + 2.0 * shear, shear


class BlendedHexagonalElasticConstants(ElasticConstants):
    """Hexagonal constants from {name: (low fit, high line)} branches in GPa, blended at x0 (2.1.4.2)."""

    def __init__(self, branches):
        self.branches = branches

    def voigt_constants(self, compositions):
        return tuple(joined_branches(compositions, self.branches[name], lattice_transition_width) * 1e9
                     for name in ELASTIC_CONSTANT_NAMES)


elastic_data_options = {"urban": BlendedHexagonalElasticConstants(urban_elastic_branches),
                        "zhang": BlendedHexagonalElasticConstants(zhang_elastic_branches),
                        "hirata_isotropic": IsotropicElasticConstants()}
chosen_elastic_constants = elastic_data_options[chosen_elastic_data]


def strained_lattice_parameter_c(composition, substrate_lattice_parameter, elastic_constants=chosen_elastic_constants):
    """c (Angstrom) of a (0001) film clamped in-plane to a substrate lattice parameter (Angstrom)."""
    a, c = lattice_parameters(composition)
    in_plane_strain = substrate_lattice_parameter / a - 1.0
    return c * (1.0 - elastic_constants.out_of_plane_coefficient(composition) * in_plane_strain)


def normal_strain_energy_density(strain_t, strain_n, strain_z, constants):
    """Elastic energy density (J/m^3) for normal strains along two basal directions and c; constants (C11, C12, C13, C33)."""
    c11, c12, c13, c33 = constants
    return (0.5 * c11 * (strain_t ** 2 + strain_n ** 2) + 0.5 * c33 * strain_z ** 2 + c12 * strain_t * strain_n
            + c13 * (strain_t + strain_n) * strain_z)

### 4.1.3 Critical thickness and misfit relaxation

Energy balance of Zhang *et al.* (2013) and Holec (2008): a misfit dislocation is favoured once the misfit energy it
relieves exceeds its own energy, $b_\perp M|\epsilon|\,h_c = A\ln(h_c/r_c) + E_c$, with $b_\perp = (\sqrt3/2)a$ for the
$\tfrac13\langle11\bar23\rangle\{1\bar101\}$ slip system, the biaxial modulus $M$ of the elastic data and the isotropic
line-energy prefactor $A = \mu(a^2+c^2-\nu a^2/4)/[4\pi(1-\nu)]$ from their Hill average. Validated against Zhang's inputs in Hirata notebook 19e, where Zhang's
"2 nm" at $x = 0.375$ is not reproduced (1.2 nm). Above $h_c$ the film keeps the fraction $f(h, h_c)$ (2.1.5) of its
coherent energy; a coherent film keeps the fraction $\sqrt f$ of its misfit.

In [ ]:
def misfit_balance(thickness_nm, released_per_nm, prefactor, core_energy, core_radius_nm):
    """Relieved misfit energy minus dislocation energy per unit length, J/m; zero at h_c."""
    return released_per_nm * thickness_nm - prefactor * math.log(thickness_nm / core_radius_nm) - core_energy


def dislocation_energy_terms(composition, substrate_lattice_parameter, elastic_constants):
    """(relieved energy per nm of thickness, line-energy prefactor A), J/m; A from the Hill-averaged moduli."""
    a, c = (float(value) for value in lattice_parameters(composition))
    youngs, poisson = (float(value) for value in elastic_constants.isotropic_moduli(composition))
    misfit = abs(substrate_lattice_parameter / a - 1.0)
    biaxial = float(elastic_constants.biaxial_modulus(composition))
    released = math.sqrt(3.0) / 2.0 * a * METRE_PER_ANGSTROM * biaxial * misfit * METRE_PER_NANOMETRE
    shear = youngs / (2.0 * (1.0 + poisson))
    prefactor = shear * (a * a + c * c - poisson * a * a / 4.0) * METRE_PER_ANGSTROM ** 2 / (4.0 * math.pi * (1.0 - poisson))
    return released, prefactor


def critical_thickness_nm(composition, core_energy, core_radius,
                          substrate_lattice_parameter=substrate_in_plane_lattice["AlN"],
                          elastic_constants=chosen_elastic_constants):
    """Equilibrium critical thickness (nm) of a (0001) film; core_energy(x) in eV/Angstrom, core_radius(x) in nm.

    The larger root of the energy balance; where none exists (dislocations favoured at any thickness) the turning
    point of the balance, which keeps h_c continuous. Then smoothly floored at the core radius, below which the
    model does not apply. Infinity for a lattice-matched film.
    """
    released, prefactor = dislocation_energy_terms(composition, substrate_lattice_parameter, elastic_constants)
    if released == 0.0:
        return math.inf
    radius = float(core_radius(composition))
    arguments = (released, prefactor, float(core_energy(composition)) * ELECTRONVOLT_PER_ANGSTROM_IN_JOULE_PER_METRE,
                 radius)
    turning_point = prefactor / released                      # minimum of the balance, nm
    thickness = turning_point
    if misfit_balance(turning_point, *arguments) < 0.0:
        upper = 2.0 * turning_point
        while misfit_balance(upper, *arguments) < 0.0:
            upper *= 2.0
        thickness = brentq(misfit_balance, turning_point, upper, args=arguments)
    width = core_radius_floor_width_nm
    return radius + width * float(np.logaddexp(0.0, (thickness - radius) / width))


@functools.lru_cache(maxsize=None)
def inverse_critical_thickness_table(core_energy, core_radius, substrate_lattice_parameter, elastic_constants):
    """1/h_c (1/nm) on a composition mesh; 0 where the film never relaxes."""
    mesh = np.linspace(0.0, 1.0, 201)
    return mesh, np.array([1.0 / critical_thickness_nm(x, core_energy, core_radius, substrate_lattice_parameter,
                                                       elastic_constants) for x in mesh])


def relaxation_factors(compositions, film_thickness_nm, core_energy, core_radius, substrate_lattice_parameter,
                       elastic_constants=chosen_elastic_constants):
    """Hirata's f(h) = (h_c/h)(1 + ln(h/h_c)) for h > h_c, else 1, with h_c(x) from the core parameters."""
    mesh, inverse = inverse_critical_thickness_table(core_energy, core_radius, float(substrate_lattice_parameter),
                                                     elastic_constants)
    ratio = film_thickness_nm * np.interp(compositions, mesh, inverse)          # h / h_c
    return np.where(ratio <= 1.0, 1.0, (1.0 + np.log(np.maximum(ratio, 1.0))) / np.maximum(ratio, 1.0))


def partially_relaxed_lattice(composition, substrate_lattice_parameter, film_thickness_nm, core_energy, core_radius,
                              elastic_constants=chosen_elastic_constants):
    """In-plane lattice imposed on a coherent film of average composition x, partly relaxed by dislocations."""
    own = float(lattice_parameters(composition)[0])
    retained = math.sqrt(float(relaxation_factors(np.array(composition), film_thickness_nm, core_energy,
                                                  core_radius, substrate_lattice_parameter, elastic_constants)))
    return own * (1.0 + retained * (substrate_lattice_parameter / own - 1.0))

## 4.2 Elastic energy of a decomposed film

A decomposed film consists of domains $i$ with compositions $x_i$, mole fractions $f_i$ and stress-free lattices
$a(x_i)$, $c(x_i)$. Each model returns $\sum_i f_i V_m(x_i)\,w_i$ in J per mol of formula unit of the whole film, for
arrays of shape (..., domains). Domains are flat layers much thicker than the interface; no interface energy.

| model | geometry | constraint |
|---|---|---|
| `IncoherentDomains` | dislocated interfaces | none |
| `CoherentLaminate` | free-standing (bulk, or a film fully relaxed from its substrate) | domains share the lattice in the interface plane; the interface orientation $\theta$ (from $c$) is a variable |
| `ClampedLayered` | film on a substrate, interfaces parallel to it | every layer clamped in-plane to $a_\mathrm{sub}$ |
| `ClampedLateral` | film on a substrate, columns | domains share the lattice along the interface; their average in-plane lattice is clamped |
| `RelaxedClampedDomains` | film on a substrate, partly relaxed; Hirata's film model | each domain clamped and relaxed separately by $f(h, h_c(x_i))$; no strain between domains |

### 4.2.1 Domain bookkeeping and the common interface

In [ ]:
def domain_properties(compositions, fractions):
    """Lattice misfits relative to the average-composition lattice, and fraction * molar volume."""
    compositions = np.asarray(compositions, dtype=float)
    fractions = np.asarray(fractions, dtype=float)
    reference_a, reference_c = lattice_parameters(np.sum(fractions * compositions, axis=-1))
    a, c = lattice_parameters(compositions)
    misfit_a = a / reference_a[..., None] - 1.0
    misfit_c = c / reference_c[..., None] - 1.0
    return misfit_a, misfit_c, fractions * molar_volume(compositions), reference_a


def energy_at_angle(angle, model, compositions, fractions):
    """Scalar elastic energy of one configuration at one interface angle, for 1-D minimisation."""
    return float(np.squeeze(model.energy(compositions, fractions, angle)))


class ElasticModel:
    """Elastic energy (J/mol-f.u.) of a set of domains: compositions, fractions of shape (..., domains) -> (...).

    `angle` is the interface orientation (radians from c) for models that have one; others ignore it.
    """

    def energy(self, compositions, fractions, angle=None):
        raise NotImplementedError

    def energy_and_best_angle(self, compositions, fractions):
        """Energy at the softest orientation on the interface_angles grid, and that angle (None if no orientation)."""
        return self.energy(compositions, fractions), None

    def minimum_energy(self, compositions, fractions):
        """Energy at the softest orientation, found continuously; for a single configuration."""
        return self.energy(compositions, fractions)

### 4.2.2 Incoherent domains

In [ ]:
def no_elastic_energy(compositions, fractions):
    """Incoherent domains: zero, J/mol-f.u."""
    return np.zeros(np.shape(compositions)[:-1])


class IncoherentDomains(ElasticModel):
    def energy(self, compositions, fractions, angle=None):
        return no_elastic_energy(compositions, fractions)

### 4.2.3 Coherent laminate, free-standing

For an interface normal at angle $\theta$ from $c$, the in-plane misfits are $\epsilon_a$ and
$\epsilon_a\cos^2\theta + \epsilon_c\sin^2\theta$; normal and out-of-plane shear stresses relax (plane stress, with the
$\theta$-dependent stiffness of 4.1.2), and the shared in-plane lattice minimises the total, a $2\times2$ solve. Reduces
to Cahn–Larché for constant moduli and linear misfit.

In [ ]:
def coherent_laminate_energy_by_angle(compositions, fractions, elastic_constants, angles=interface_angles):
    """Free-standing coherent laminate, J/mol-f.u., for each interface angle (radians from c): (..., angles)."""
    misfit_a, misfit_c, weights, _ = domain_properties(compositions, fractions)
    tilted = misfit_a[..., None] * np.cos(angles) ** 2 + misfit_c[..., None] * np.sin(angles) ** 2
    misfits = np.stack([np.broadcast_to(misfit_a[..., None], tilted.shape), tilted], axis=-1)
    stiffness = elastic_constants.plane_stress_stiffness(np.asarray(compositions, dtype=float), angles)
    weighted = weights[..., None, None, None] * stiffness                        # (..., domains, angles, 2, 2)
    load = np.einsum("...daij,...daj->...ai", weighted, misfits)                # (..., angles, 2)
    shared = np.linalg.solve(weighted.sum(axis=-4), load[..., None])[..., 0]
    elastic = shared[..., None, :, :] - misfits                                 # (..., domains, angles, 2)
    return 0.5 * np.einsum("...dai,...daij,...daj->...a", elastic, weighted, elastic)


class CoherentLaminate(ElasticModel):
    """Free-standing coherent laminate; angle=None takes the softest orientation on the angle grid."""

    def __init__(self, elastic_constants=chosen_elastic_constants):
        self.elastic_constants = elastic_constants

    def energy(self, compositions, fractions, angle=None):
        angles = interface_angles if angle is None else np.array([angle])
        return coherent_laminate_energy_by_angle(compositions, fractions, self.elastic_constants, angles).min(axis=-1)

    def energy_and_best_angle(self, compositions, fractions):
        by_angle = coherent_laminate_energy_by_angle(compositions, fractions, self.elastic_constants)
        return by_angle.min(axis=-1), interface_angles[np.argmin(by_angle, axis=-1)]

    def minimum_energy(self, compositions, fractions):
        start = float(np.squeeze(self.energy_and_best_angle(compositions, fractions)[1]))
        step = interface_angles[1] - interface_angles[0]
        bounds = (max(start - step, 0.0), min(start + step, 0.5 * math.pi))
        result = minimize_scalar(energy_at_angle, bounds=bounds, args=(self, compositions, fractions),
                                 method="bounded", options={"xatol": 1e-4})
        return min(result.fun, float(np.squeeze(self.energy(compositions, fractions, start))))

### 4.2.4 Coherent domains in a clamped film

Layered: each layer biaxially clamped to $a_\mathrm{sub}$. Lateral: the film normal is $c$, the interface normal lies in
the basal plane; along the interface every domain is at $a_\mathrm{sub}$, along $c$ the domains share one strain with
zero average stress, along the interface normal their fraction-weighted strain is fixed by the substrate — a
$4\times4$ Lagrange solve.

In [ ]:
def clamping_energies(compositions, substrate_lattice_parameter, elastic_constants):
    """Biaxial clamping energy of each domain on its own, J per mol of formula unit of that domain."""
    compositions = np.asarray(compositions, dtype=float)
    a, _ = lattice_parameters(compositions)
    strain = np.asarray(substrate_lattice_parameter)[..., None] / a - 1.0
    return molar_volume(compositions) * elastic_constants.biaxial_modulus(compositions) * strain ** 2


def clamped_layered_energy(compositions, fractions, substrate_lattice_parameter, elastic_constants):
    """Every domain biaxially clamped to the substrate (basal interfaces); J/mol-f.u."""
    energies = clamping_energies(compositions, substrate_lattice_parameter, elastic_constants)
    return np.sum(np.asarray(fractions) * energies, axis=-1)


def lateral_constrained_system(weights, constants, misfit_a, misfit_c, strain_t, fractions):
    """Lagrange system for (u_B, u_C, w, multiplier): normal strains of the two domains and shared c strain.

    constants: (C11, C12, C13, C33) per domain; t, n are the basal directions along and normal to the interface.
    """
    shape = weights.shape[:-1]
    matrix = np.zeros(shape + (4, 4))
    right = np.zeros(shape + (4,))
    for domain in (0, 1):
        weight, elastic_t = weights[..., domain], strain_t[..., domain]
        c11, c12, c13, c33 = (value[..., domain] for value in constants)
        matrix[..., domain, domain] = weight * c11
        matrix[..., domain, 2] = matrix[..., 2, domain] = weight * c13
        matrix[..., 2, 2] += weight * c33
        right[..., domain] = weight * (c11 * misfit_a[..., domain] - c12 * elastic_t + c13 * misfit_c[..., domain])
        right[..., 2] += weight * (c33 * misfit_c[..., domain] - c13 * (elastic_t - misfit_a[..., domain]))
        matrix[..., domain, 3] = matrix[..., 3, domain] = fractions[..., domain]
    return matrix, right


def clamped_lateral_energy(compositions, fractions, substrate_lattice_parameter, elastic_constants):
    """Columnar domains on a clamping substrate (interfaces normal to the film); J/mol-f.u.

    compositions, fractions: shape (..., 2); a single domain falls back to the layered form.
    """
    compositions = np.asarray(compositions, dtype=float)
    fractions = np.asarray(fractions, dtype=float)
    if compositions.shape[-1] == 1:
        return clamped_layered_energy(compositions, fractions, substrate_lattice_parameter, elastic_constants)
    misfit_a, misfit_c, weights, reference_a = domain_properties(compositions, fractions)
    substrate_strain = np.asarray(substrate_lattice_parameter) / reference_a - 1.0
    strain_t = substrate_strain[..., None] - misfit_a
    constants = elastic_constants.voigt_constants(compositions)[:4]
    matrix, right = lateral_constrained_system(weights, constants, misfit_a, misfit_c, strain_t, fractions)
    right[..., 3] = substrate_strain
    solution = np.linalg.solve(matrix, right[..., None])[..., 0]
    strain_n = solution[..., :2] - misfit_a
    strain_z = solution[..., 2:3] - misfit_c
    density = normal_strain_energy_density(strain_t, strain_n, strain_z, constants)
    return np.sum(weights * density, axis=-1)


class ClampedLayered(ElasticModel):
    """Every domain biaxially clamped to a substrate in-plane lattice parameter (Angstrom)."""

    def __init__(self, substrate_lattice_parameter, elastic_constants=chosen_elastic_constants):
        self.substrate_lattice_parameter = substrate_lattice_parameter
        self.elastic_constants = elastic_constants

    def energy(self, compositions, fractions, angle=None):
        return clamped_layered_energy(compositions, fractions, self.substrate_lattice_parameter,
                                      self.elastic_constants)


class ClampedLateral(ClampedLayered):
    """Columnar domains whose average in-plane lattice is clamped to the substrate."""

    def energy(self, compositions, fractions, angle=None):
        return clamped_lateral_energy(compositions, fractions, self.substrate_lattice_parameter,
                                      self.elastic_constants)

### 4.2.5 Domains relaxed separately on a substrate (Hirata)

In [ ]:
class RelaxedClampedDomains(ElasticModel):
    """Domains clamped to the substrate independently, each keeping f(h, h_c(x)) of its clamping energy.

    Hirata's film model: misfit dislocations relax every domain separately; no strain between domains.
    film_thickness_nm=None: fully clamped (thinnest-film limit).
    """

    def __init__(self, substrate_lattice_parameter, film_thickness_nm, core_energy, core_radius,
                 elastic_constants=chosen_elastic_constants):
        self.substrate_lattice_parameter = substrate_lattice_parameter
        self.relaxation = (film_thickness_nm, core_energy, core_radius)
        self.elastic_constants = elastic_constants

    def energy(self, compositions, fractions, angle=None):
        energies = clamping_energies(compositions, self.substrate_lattice_parameter, self.elastic_constants)
        if self.relaxation[0] is not None:
            energies = energies * relaxation_factors(compositions, *self.relaxation,
                                                     self.substrate_lattice_parameter, self.elastic_constants)
        return np.sum(np.asarray(fractions) * energies, axis=-1)

### 4.2.6 Anharmonic elasticity — not implemented

Planned flavour: direct DFT epitaxial energies per interface orientation in place of linear elasticity. Hexagonal
$C_{ij}(x)$ are now an elastic data option (2.2.3).

## 4.3 Decomposition and stability at one composition

**Decomposition (binodal).** The film stays homogeneous or splits into two domains $x_B < \bar x < x_C$ with fraction
$f = (x_C-\bar x)/(x_C-x_B)$ of $x_B$:
$\Delta G_\mathrm{decomp} = [G(\bar x) + E_\mathrm{el}(\bar x)] - \min_{x_B,x_C}[f G(x_B) + (1-f) G(x_C) + E_\mathrm{el}(x_B, x_C, f)]$.
$\Delta G_\mathrm{decomp} > 0$: the film decomposes; the binodal is where it becomes zero. For incoherent domains this is
the common tangent; for coherent ones it is not, because $E_\mathrm{el}$ is not linear in $f$. The variables are $x_B$,
$x_C$ and, for the laminate, $\theta$; a gradient method is started from the best pairs of a grid spaced finely near
$\bar x$, from well-separated pairs, and from a small split, which guarantees that a film inside the spinodal is
found to decompose.

**Stability (spinodal).** The same comparison for a split $\bar x \mp \delta$ at $f = 1/2$ gives an effective curvature
$G''_\mathrm{eff} = 2[\ldots]/\delta^2$; the film is unstable where it is negative. For the laminate it is Cahn's
$\partial^2G/\partial x^2 + 2\eta^2 Y V_m$ minimised over orientation. Two-domain states only.

### 4.3.1 Search

In [ ]:
@dataclasses.dataclass
class DecompositionResult:
    """Best two-domain state at one average composition; energies in J/mol-f.u."""
    average_composition: float
    driving_force: float                  # > 0: decomposition lowers G; 0: stays homogeneous
    domain_compositions: tuple            # (x_B, x_C); both equal to the average if homogeneous
    fraction_of_lower_domain: float       # f of x_B
    interface_angle_degrees: float = None  # coherent laminate only; from the c axis
    geometry: str = None                   # coherent clamped only: "layered" or "lateral"


def pair_arrays(lower, upper, average):
    """Domain compositions and lever-rule fractions for arrays of pairs: shapes (..., 2)."""
    lower, upper = np.asarray(lower, dtype=float), np.asarray(upper, dtype=float)
    fraction = (upper - average) / (upper - lower)
    return np.stack([lower, upper], axis=-1), np.stack([fraction, 1.0 - fraction], axis=-1)


def chemical_part(gibbs_energy, temperature, compositions, fractions):
    """Fraction-weighted G of the domains, J/mol-f.u."""
    return np.sum(fractions * gibbs_energy(compositions, temperature), axis=-1)


def homogeneous_free_energy(gibbs_energy, temperature, average, elastic_model):
    """G + E_el of the undecomposed film at the average composition, J/mol-f.u."""
    elastic = elastic_model.minimum_energy(np.array([[average]]), np.array([[1.0]]))
    return float(gibbs_energy(np.array(average), temperature)) + float(np.squeeze(elastic))


def pair_free_energy(variables, gibbs_energy, temperature, average, elastic_model):
    """f G(x_B) + (1-f) G(x_C) + E_el for variables (x_B, x_C[, angle]) at one average composition."""
    compositions, fractions = pair_arrays(variables[0], variables[1], average)
    elastic = elastic_model.energy(compositions[None], fractions[None], *variables[2:])
    return float(chemical_part(gibbs_energy, temperature, compositions, fractions) + elastic[0])


def partner_axis(average, end):
    """Grid of partner compositions between the average and an end (0 or 1), denser near the average."""
    steps = np.linspace(1.0, 0.0, partner_grid_points, endpoint=False) ** 2
    axis = average + (end - average) * steps
    small = average + np.sign(end - average) * min(small_split, 0.5 * abs(end - average))
    return np.clip(np.append(axis, small), composition_margin, 1.0 - composition_margin)


def starting_points(gibbs_energy, temperature, average, elastic_model):
    """Starting points (x_B, x_C[, angle]): best grid pairs overall, best well-separated pairs, small split."""
    lower, upper = (axis.ravel() for axis in np.meshgrid(partner_axis(average, 0.0), partner_axis(average, 1.0),
                                                       indexing="ij"))
    compositions, fractions = pair_arrays(lower, upper, average)
    elastic, angles = elastic_model.energy_and_best_angle(compositions, fractions)
    order = np.argsort(chemical_part(gibbs_energy, temperature, compositions, fractions) + elastic)
    separated = order[upper[order] - lower[order] > separation]
    chosen = list(order[:refinement_starts]) + list(separated[:refinement_starts]) + [len(lower) - 1]
    return [(lower[i], upper[i]) + (() if angles is None else (angles[i],)) for i in dict.fromkeys(chosen)]


def variable_bounds(average, has_angle):
    """Bounds keeping x_B < average < x_C, and the angle within [0, 90 degrees]."""
    bounds = [(composition_margin, average - composition_margin), (average + composition_margin,
                                                                   1.0 - composition_margin)]
    return bounds + ([(0.0, 0.5 * math.pi)] if has_angle else [])


def decomposition(gibbs_energy, temperature, average_composition, elastic_model, method=refinement_method):
    """Best two-domain state of a film of average composition x (0 < x < 1); returns DecompositionResult.

    gibbs_energy(x, T): J/mol-f.u., vectorised in x. elastic_model: a section-17 ElasticModel.
    """
    average = float(average_composition)
    homogeneous = homogeneous_free_energy(gibbs_energy, temperature, average, elastic_model)
    best = None
    for start in starting_points(gibbs_energy, temperature, average, elastic_model):
        result = minimize(pair_free_energy, start, args=(gibbs_energy, temperature, average, elastic_model),
                          method=method, bounds=variable_bounds(average, len(start) == 3))
        best = result if best is None or result.fun < best.fun else best
    if best.fun >= homogeneous:
        return DecompositionResult(average, 0.0, (average, average), 1.0)
    lower, upper = best.x[:2]
    angle = math.degrees(best.x[2]) if len(best.x) == 3 else None
    return DecompositionResult(average, homogeneous - best.fun, (lower, upper),
                               (upper - average) / (upper - lower), angle)


def stability(gibbs_energy, temperature, average_composition, elastic_model,
              amplitude=fluctuation_amplitude):
    """Effective curvature (J/mol-f.u.) against a small +-amplitude two-domain split; < 0 is unstable."""
    average = float(average_composition)
    amplitude = min(amplitude, 0.5 * average, 0.5 * (1.0 - average))
    compositions, fractions = pair_arrays([average - amplitude], [average + amplitude], average)
    split = (chemical_part(gibbs_energy, temperature, compositions, fractions)[0]
             + float(np.squeeze(elastic_model.minimum_energy(compositions, fractions))))
    homogeneous = homogeneous_free_energy(gibbs_energy, temperature, average, elastic_model)
    return 2.0 * (split - homogeneous) / amplitude ** 2

### 4.3.2 Functions per case

| case | decomposition (binodal) | stability (spinodal) |
|---|---|---|
| free-standing, incoherent | `incoherent_decomposition` | `incoherent_stability` |
| free-standing, coherent | `coherent_decomposition` | `coherent_stability` |
| on a substrate, domains relaxed separately (Hirata) | `incoherent_clamped_decomposition` | `incoherent_clamped_stability` |
| on a substrate, coherent domains | `coherent_clamped_decomposition` | `coherent_clamped_stability` |

All take `gibbs_energy(x, T)`, `T` and $\bar x$; the clamped ones also the substrate lattice parameter (Å). Coherent
clamped films take `geometry = "layered"`, `"lateral"` or `"lowest"`. Those with elastic energy take
`elastic_constants` (default: the choice of 2.2.3). For a fully clamped film the layered geometry
and Hirata's case coincide.

In [ ]:
CLAMPED_GEOMETRIES = {"layered": ClampedLayered, "lateral": ClampedLateral}


def incoherent_decomposition(gibbs_energy, temperature, average_composition):
    """Free-standing, dislocated domains, each at its own lattice; returns DecompositionResult."""
    return decomposition(gibbs_energy, temperature, average_composition, IncoherentDomains())


def coherent_decomposition(gibbs_energy, temperature, average_composition, elastic_constants=chosen_elastic_constants):
    """Free-standing coherent domains, softest interface orientation; returns DecompositionResult."""
    return decomposition(gibbs_energy, temperature, average_composition, CoherentLaminate(elastic_constants))


def incoherent_clamped_decomposition(gibbs_energy, temperature, average_composition, substrate_lattice_parameter,
                                     film_thickness_nm, core_energy, core_radius,
                                     elastic_constants=chosen_elastic_constants):
    """Film on a substrate, domains relaxed separately (Hirata); film_thickness_nm=None: fully clamped."""
    model = RelaxedClampedDomains(substrate_lattice_parameter, film_thickness_nm, core_energy, core_radius,
                                  elastic_constants)
    return decomposition(gibbs_energy, temperature, average_composition, model)


def coherent_clamped_decomposition(gibbs_energy, temperature, average_composition, substrate_lattice_parameter,
                                   geometry="lowest", elastic_constants=chosen_elastic_constants):
    """Coherent domains in a film clamped to a substrate lattice parameter (Angstrom); DecompositionResult."""
    if geometry != "lowest":
        result = decomposition(gibbs_energy, temperature, average_composition,
                               CLAMPED_GEOMETRIES[geometry](substrate_lattice_parameter, elastic_constants))
        return dataclasses.replace(result, geometry=geometry if result.driving_force > 0.0 else None)
    return max((coherent_clamped_decomposition(gibbs_energy, temperature, average_composition,
                                               substrate_lattice_parameter, option, elastic_constants)
                for option in CLAMPED_GEOMETRIES), key=operator.attrgetter("driving_force"))


def incoherent_stability(gibbs_energy, temperature, average_composition):
    """d2G/dx2, J/mol-f.u.; < 0 inside the chemical spinodal."""
    return stability(gibbs_energy, temperature, average_composition, IncoherentDomains())


def coherent_stability(gibbs_energy, temperature, average_composition, elastic_constants=chosen_elastic_constants):
    """Cahn coherent-spinodal curvature, softest orientation, J/mol-f.u."""
    return stability(gibbs_energy, temperature, average_composition, CoherentLaminate(elastic_constants))


def incoherent_clamped_stability(gibbs_energy, temperature, average_composition, substrate_lattice_parameter,
                                 film_thickness_nm, core_energy, core_radius,
                                 elastic_constants=chosen_elastic_constants):
    """Spinodal curvature of a film whose domains relax separately (Hirata), J/mol-f.u."""
    model = RelaxedClampedDomains(substrate_lattice_parameter, film_thickness_nm, core_energy, core_radius,
                                  elastic_constants)
    return stability(gibbs_energy, temperature, average_composition, model)


def coherent_clamped_stability(gibbs_energy, temperature, average_composition, substrate_lattice_parameter,
                               geometry="lowest", elastic_constants=chosen_elastic_constants):
    """Spinodal curvature of a clamped coherent film, J/mol-f.u.; 'lowest' takes the less stable geometry."""
    if geometry != "lowest":
        return stability(gibbs_energy, temperature, average_composition,
                         CLAMPED_GEOMETRIES[geometry](substrate_lattice_parameter, elastic_constants))
    return min(coherent_clamped_stability(gibbs_energy, temperature, average_composition,
                                          substrate_lattice_parameter, option, elastic_constants)
               for option in CLAMPED_GEOMETRIES)

## 4.4 A case: one physical situation

`film_case` binds everything that defines a situation — coherent or not, the substrate's in-plane lattice parameter
(none, a number, or a function of $\bar x$ for a graded substrate), the film thickness, the elastic data and the
dislocation-core functions. It returns a `FilmCase` whose `decomposition(G, T, x)` and `stability(G, T, x)` are what
section 4.5 maps over temperature and composition.

| arguments | situation |
|---|---|
| no substrate | free-standing (bulk, or a film fully relaxed from its substrate); thickness ignored |
| substrate, `film_thickness_nm=None` | fully clamped (thinner than $h_c$) |
| substrate, thickness | misfit partly relaxed by dislocations, $h_c(x)$ from the core functions. Incoherent: each domain relaxed separately (Hirata); coherent: the film's imposed in-plane lattice relaxes towards its own |

In [ ]:
@dataclasses.dataclass(frozen=True)
class FilmCase:
    """One situation with every parameter bound: decomposition(G, T, x) and stability(G, T, x)."""
    description: str
    decomposition: object
    stability: object


def substrate_at(substrate_lattice_parameter, composition):
    """Substrate lattice parameter for one average composition: a number or a function of x."""
    if callable(substrate_lattice_parameter):
        return substrate_lattice_parameter(composition)
    return substrate_lattice_parameter


def on_substrate(function, substrate_lattice_parameter, gibbs_energy, temperature, composition):
    """Call a clamped decomposition or stability function with the substrate lattice parameter at this x."""
    return function(gibbs_energy, temperature, composition, substrate_at(substrate_lattice_parameter, composition))


def relaxed_substrate_lattice(composition, substrate_lattice_parameter, film_thickness_nm, core_energy, core_radius,
                              elastic_constants):
    """In-plane lattice imposed on a coherent film above h_c: the substrate's, relaxed towards the film's own."""
    return partially_relaxed_lattice(composition, substrate_at(substrate_lattice_parameter, composition),
                                     film_thickness_nm, core_energy, core_radius, elastic_constants)


def elastic_data_name(elastic_constants):
    return next((name for name, option in elastic_data_options.items() if option is elastic_constants), "custom")


def case_description(coherent, substrate_lattice_parameter, film_thickness_nm, elastic_constants, core_energy):
    """Readable statement of a case's conditions."""
    kind = "coherent" if coherent else "incoherent"
    if substrate_lattice_parameter is None:
        film = "free-standing"
    else:
        substrate = ("a graded substrate a(x)" if callable(substrate_lattice_parameter)
                     else f"a = {substrate_lattice_parameter:.4f} Angstrom")
        film = f"on {substrate}, " + ("fully clamped" if film_thickness_nm is None else
                                      f"{film_thickness_nm:g} nm, h_c with {core_energy.__name__}")
    return f"{kind}, {film}; elastic data '{elastic_data_name(elastic_constants)}'"


def free_standing_functions(coherent, elastic_constants):
    """(decomposition, stability) of a free-standing film."""
    if not coherent:
        return incoherent_decomposition, incoherent_stability
    return (functools.partial(coherent_decomposition, elastic_constants=elastic_constants),
            functools.partial(coherent_stability, elastic_constants=elastic_constants))


def film_on_substrate_functions(coherent, substrate_lattice_parameter, film_thickness_nm, elastic_constants,
                                core_energy, core_radius):
    """(decomposition, stability) of a film on a substrate, with the substrate bound."""
    if coherent:
        functions = (coherent_clamped_decomposition, coherent_clamped_stability)
        options = dict(elastic_constants=elastic_constants)
        if film_thickness_nm is not None:
            substrate_lattice_parameter = functools.partial(
                relaxed_substrate_lattice, substrate_lattice_parameter=substrate_lattice_parameter,
                film_thickness_nm=film_thickness_nm, core_energy=core_energy, core_radius=core_radius,
                elastic_constants=elastic_constants)
    else:
        functions = (incoherent_clamped_decomposition, incoherent_clamped_stability)
        options = dict(film_thickness_nm=film_thickness_nm, core_energy=core_energy, core_radius=core_radius,
                       elastic_constants=elastic_constants)
    return tuple(functools.partial(on_substrate, functools.partial(function, **options), substrate_lattice_parameter)
                 for function in functions)


def film_case(coherent, substrate_lattice_parameter=None, film_thickness_nm=None,
              elastic_constants=chosen_elastic_constants, core_energy=chosen_core_energy, core_radius=chosen_core_radius):
    """A FilmCase. substrate_lattice_parameter: None (free-standing), Angstrom, or a function of x.

    film_thickness_nm: None (fully clamped) or nm; ignored when free-standing.
    """
    if substrate_lattice_parameter is None:
        functions = free_standing_functions(coherent, elastic_constants)
        film_thickness_nm = None
    else:
        functions = film_on_substrate_functions(coherent, substrate_lattice_parameter, film_thickness_nm,
                                                elastic_constants, core_energy, core_radius)
    description = case_description(coherent, substrate_lattice_parameter, film_thickness_nm, elastic_constants,
                                   core_energy)
    return FilmCase(description, *functions)

## 4.5 Phase boundaries, driving-force maps and critical temperatures

Each function takes a `FilmCase` and returns plain arrays in a small result object, so results can be stored, replotted
and analysed. Temperatures are independent of each other and are computed in `worker_processes` parallel processes
(2.2.6).

- **Binodal:** the driving force on a coarse scan of $\bar x$, every crossing of the threshold located by root-finding.
  A film already inside the spinodal is inside the binodal, so its search is skipped.
- **Spinodal:** $G''_\mathrm{eff}$ on a grid.
- **Critical temperature:** bisection on whether any composition is spinodal-unstable.

In [ ]:
@dataclasses.dataclass
class PhaseBoundaries:
    """Binodal points and the spinodal curvature map of one case."""
    description: str
    binodal: np.ndarray                  # (points, 2): temperature (K), average composition
    spinodal_temperatures: np.ndarray    # K
    spinodal_compositions: np.ndarray
    spinodal_curvature: np.ndarray       # (temperatures, compositions), J/mol-f.u.; < 0 unstable


@dataclasses.dataclass
class DecompositionMap:
    """Best two-domain states of one case on a (temperature, composition) grid."""
    description: str
    temperatures: np.ndarray
    compositions: np.ndarray
    results: list                        # [temperature index][composition index] -> DecompositionResult

    def driving_forces(self):
        """(temperatures, compositions), J/mol-f.u."""
        return np.array([[result.driving_force for result in row] for row in self.results])


def parallel_map(function, items):
    """[function(item) for item in items], in worker_processes forked processes."""
    if worker_processes <= 1:
        return [function(item) for item in items]
    context = multiprocessing.get_context("fork")
    with concurrent.futures.ProcessPoolExecutor(worker_processes, mp_context=context) as pool:
        return list(pool.map(function, items))


def threshold_excess(composition, case, gibbs_energy, temperature):
    return case.decomposition(gibbs_energy, temperature, composition).driving_force - decomposition_threshold


def is_decomposing(composition, case, gibbs_energy, temperature):
    """True if the film decomposes; the cheap spinodal test first."""
    if case.stability(gibbs_energy, temperature, composition) < 0.0:
        return True
    return threshold_excess(composition, case, gibbs_energy, temperature) > 0.0


def binodal_compositions(case, gibbs_energy, temperature, scan=binodal_scan):
    """Average compositions at which the driving force crosses the threshold, at one temperature."""
    arguments = (case, gibbs_energy, temperature)
    inside = np.array([is_decomposing(x, *arguments) for x in scan])
    crossings = np.flatnonzero(inside[1:] != inside[:-1])
    return [brentq(threshold_excess, scan[i], scan[i + 1], args=arguments, xtol=1e-4) for i in crossings]


def curvature_row(case, gibbs_energy, compositions, temperature):
    """Spinodal curvature at one temperature over the compositions, J/mol-f.u."""
    return np.array([case.stability(gibbs_energy, temperature, x) for x in compositions])


def decomposition_at(case, gibbs_energy, temperature_and_composition):
    temperature, composition = temperature_and_composition
    return case.decomposition(gibbs_energy, temperature, composition)


def compute_phase_boundaries(case, gibbs_energy=wurtzite_gibbs_energy, temperatures=binodal_temperatures,
                             spinodal_grid=(spinodal_temperatures, spinodal_compositions)):
    """Binodal at each temperature and the spinodal curvature map of one case; returns PhaseBoundaries."""
    case.stability(gibbs_energy, temperatures[0], 0.3)      # fills the h_c caches once, before the processes fork
    rows = parallel_map(functools.partial(binodal_compositions, case, gibbs_energy), temperatures)
    binodal = np.array([(temperature, x) for temperature, row in zip(temperatures, rows) for x in row]).reshape(-1, 2)
    curvature = np.array(parallel_map(functools.partial(curvature_row, case, gibbs_energy, spinodal_grid[1]),
                                      spinodal_grid[0]))
    return PhaseBoundaries(case.description, binodal, *spinodal_grid, curvature)


def compute_decomposition_map(case, temperatures, compositions, gibbs_energy=wurtzite_gibbs_energy):
    """Best two-domain state at every (temperature, composition); returns DecompositionMap."""
    temperatures = np.atleast_1d(np.asarray(temperatures, dtype=float))
    compositions = np.asarray(compositions, dtype=float)
    case.stability(gibbs_energy, temperatures[0], 0.3)
    points = [(temperature, x) for temperature in temperatures for x in compositions]
    results = parallel_map(functools.partial(decomposition_at, case, gibbs_energy), points)
    rows = [results[index * len(compositions):(index + 1) * len(compositions)] for index in range(len(temperatures))]
    return DecompositionMap(case.description, temperatures, compositions, rows)


def critical_temperature(case, compositions, gibbs_energy=wurtzite_gibbs_energy, low=100.0, high=9000.0,
                         iterations=16):
    """Highest T at which the case is spinodal-unstable somewhere on the composition grid; (T, x).

    (nan, nan): stable at `low`; (inf, nan): still unstable at `high`.
    """
    unstable = functools.partial(curvature_row, case, gibbs_energy, compositions)
    if not np.any(unstable(low) < 0.0):
        return float("nan"), float("nan")
    if np.any(unstable(high) < 0.0):
        return float("inf"), float("nan")
    for _ in range(iterations):
        middle = 0.5 * (low + high)
        low, high = (middle, high) if np.any(unstable(middle) < 0.0) else (low, middle)
    return low, float(compositions[int(np.argmin(unstable(low)))])


def critical_temperature_of(case_and_compositions):
    return critical_temperature(*case_and_compositions)


def compute_critical_temperatures(cases, compositions=critical_compositions):
    """{name: ((T, x) over all compositions, (T, x) on the buckled branch x < x0 - 0.05)}, cases in parallel."""
    buckled = compositions[compositions < TRANSITION_COMPOSITION - 0.05]
    tasks = [(case, grid) for case in cases.values() for grid in (compositions, buckled)]
    results = parallel_map(critical_temperature_of, tasks)
    return {name: (results[2 * index], results[2 * index + 1]) for index, name in enumerate(cases)}

## 4.6 Substrate strain as an interaction-parameter shift

Hirata's clamped energy fitted on the buckled branch to a line plus $x(1-x)[\Delta L^{(0)} + \Delta L^{(1)}(1-2x)]$. A
near-exact fit means a strain term and a change of $L^{(0)}$ cannot be told apart from one measured boundary.

In [ ]:
strain_fit_compositions = np.linspace(0.0, 0.45, 46)    # buckled branch, film compositions


def substrate_strain_energy(composition, substrate_lattice_parameter, film_thickness_nm=None,
                            elastic_constants=chosen_elastic_constants):
    """Hirata's clamped-film energy with relaxation factor f(h, h_c(x)) (None: fully clamped), J/mol-f.u."""
    compositions = np.atleast_1d(np.asarray(composition, dtype=float))
    energy = clamping_energies(compositions[..., None], substrate_lattice_parameter, elastic_constants)[..., 0]
    if film_thickness_nm is not None:
        energy = energy * relaxation_factors(compositions, film_thickness_nm, chosen_core_energy, chosen_core_radius,
                                             substrate_lattice_parameter, elastic_constants)
    return energy


def strain_as_interaction_parameters(substrate_lattice_parameter, film_thickness_nm):
    """(dL0, dL1) in J/mol-f.u. and the relative rms residual of the fit, 0 <= x <= 0.45."""
    x = strain_fit_compositions
    energies = substrate_strain_energy(x, substrate_lattice_parameter, film_thickness_nm)
    basis = np.stack([np.ones_like(x), x, x * (1.0 - x), x * (1.0 - x) * (1.0 - 2.0 * x)], axis=1)
    coefficients, *_ = np.linalg.lstsq(basis, energies, rcond=None)
    excess = energies - basis[:, :2] @ coefficients[:2]
    residual = energies - basis @ coefficients
    return coefficients[2:], math.sqrt(np.mean(residual ** 2)) / np.max(np.abs(excess))

## 4.7 Interface for the defect model

The free energy of the **homogeneous** film, relative to the unstrained AlN–ScN wurtzite endmembers, J/mol-f.u.; with a
substrate it includes the biaxial clamping energy.

In [ ]:
def homogeneous_film_gibbs_energy(gibbs_energy, temperature, composition, substrate_lattice_parameter=None,
                                  elastic_constants=chosen_elastic_constants):
    """G of an undecomposed film; substrate_lattice_parameter: None (free) or Angstrom."""
    composition = np.asarray(composition, dtype=float)
    energy = gibbs_energy(composition, temperature)
    if substrate_lattice_parameter is None:
        return energy
    strain = substrate_strain_energy(composition, substrate_lattice_parameter, elastic_constants=elastic_constants)
    return energy + strain.reshape(np.shape(energy))

# 5. Auxiliary: standard cases and plotting helpers

## 5.1 The six standard cases

Used for the critical-temperature tables of 6.2–6.3; the phase diagrams of 6.4 spell out their cases in place.

In [ ]:
def standard_cases(elastic_constants):
    """{label: FilmCase}: free-standing, fully clamped on AlN and a real_film_thickness_nm film on AlN."""
    on_aluminium_nitride = dict(substrate_lattice_parameter=aluminium_nitride, elastic_constants=elastic_constants)
    micron = dict(on_aluminium_nitride, film_thickness_nm=real_film_thickness_nm)
    return {
        "free-standing, incoherent": film_case(coherent=False, elastic_constants=elastic_constants),
        "free-standing, coherent": film_case(coherent=True, elastic_constants=elastic_constants),
        "fully clamped on AlN, incoherent": film_case(coherent=False, **on_aluminium_nitride),
        "fully clamped on AlN, coherent": film_case(coherent=True, **on_aluminium_nitride),
        "1 um on AlN, incoherent": film_case(coherent=False, **micron),
        "1 um on AlN, coherent": film_case(coherent=True, **micron),
    }

## 5.2 Plotting and printing

Every function here takes computed results (section 4.5 objects) and only draws or prints them.

In [ ]:
CASE_COLOURS = ("k", "tab:red", "tab:blue", "tab:green", "tab:purple", "tab:orange")


def common_conditions():
    corrections = " + ".join(chemical_corrections) or "none"
    return f"Hirata G, chemical corrections: {corrections}; transition width {lattice_transition_width}"


def print_conditions(results):
    for result in results:
        print(f"Conditions: {result.description}")
    print(f"            {common_conditions()}")


def short_label(result):
    """'coherent' or 'incoherent', plus 'free-standing' if so; the full conditions are printed under the plot."""
    kind, film = result.description.split(";")[0].split(", ")[:2]
    return f"{kind}, free-standing" if film == "free-standing" else kind


def draw_boundaries(axis, boundaries, colour, label):
    """Binodal points and spinodal contour (dotted) of one PhaseBoundaries."""
    axis.plot(boundaries.binodal[:, 1], boundaries.binodal[:, 0], "o", color=colour, markersize=2.5,
              label=f"{label}, binodal")
    axis.contour(boundaries.spinodal_compositions, boundaries.spinodal_temperatures, boundaries.spinodal_curvature,
                 levels=[0.0], colors=colour, linestyles=":")
    axis.plot([], [], ":", color=colour, label=f"{label}, spinodal")


def plot_phase_diagram(*boundaries, reference=None, title=""):
    """Binodals and spinodals of any number of PhaseBoundaries; `reference` drawn in grey underneath."""
    figure, axis = plt.subplots(figsize=(7.5, 5.5))
    if reference is not None:
        draw_boundaries(axis, reference, "lightgrey", f"reference: {short_label(reference)}")
    for result, colour in zip(boundaries, CASE_COLOURS):
        draw_boundaries(axis, result, colour, short_label(result))
    axis.axhspan(growth_temperatures[0], growth_temperatures[-1], color="tab:olive", alpha=0.2)
    axis.axvline(TRANSITION_COMPOSITION, color="grey", ls="--", lw=0.8)
    axis.set_xlim(0, 1); axis.set_xlabel("Sc cation fraction $x$"); axis.set_ylabel("Temperature (K)")
    axis.set_title(f"{title} (band: growth window 300-450 C; dashed: x0)")
    axis.legend(fontsize=7, loc="upper right"); axis.grid(alpha=0.25)
    print_conditions(boundaries + ((reference,) if reference is not None else ()))
    plt.tight_layout(); plt.show()


def plot_driving_force(*maps, title=""):
    """Decomposition driving force of any number of DecompositionMaps, one line per temperature, meV/f.u."""
    figure, axis = plt.subplots(figsize=(7.5, 4.5))
    styles = (":", "-", "--", "-.")
    for result, colour in zip(maps, CASE_COLOURS):
        forces = result.driving_forces() * MILLIELECTRONVOLT_PER_JOULE_PER_MOLE
        for row, (temperature, style) in enumerate(zip(result.temperatures, styles)):
            axis.plot(result.compositions, forces[row], style, color=colour,
                      label=f"{short_label(result)}, {temperature - 273.15:.0f} C")
    axis.axvline(TRANSITION_COMPOSITION, color="grey", ls="--", lw=0.8)
    axis.set_xlabel("Sc cation fraction $x$"); axis.set_ylabel(r"$\Delta G_\mathrm{decomp}$ (meV/f.u.)")
    axis.set_title(f"Decomposition driving force: {title}"); axis.legend(fontsize=7); axis.grid(alpha=0.25)
    print_conditions(maps)
    plt.tight_layout(); plt.show()


def describe(result):
    """'driving force (x_B + x_C) geometry' in meV/f.u."""
    lower, upper = result.domain_compositions
    geometry = (f"{result.interface_angle_degrees:3.0f} deg" if result.interface_angle_degrees is not None
                else result.geometry or "")
    return f"{result.driving_force * MILLIELECTRONVOLT_PER_JOULE_PER_MOLE:6.1f} ({lower:.3f}+{upper:.3f}) {geometry}"


def print_best_states(*maps, temperature, compositions=(0.1, 0.2, 0.3, 0.4, 0.5)):
    """Driving force, domain compositions and geometry at one of the maps' temperatures."""
    print(f"Best state at {temperature - 273.15:.0f} C: driving force (meV/f.u.), domains x_B+x_C, "
          "geometry (laminate: interface-normal angle from c)")
    for result in maps:
        row = result.results[int(np.argmin(np.abs(result.temperatures - temperature)))]
        print(f"  {result.description.split(';')[0]}")
        for composition in compositions:
            print(f"    x = {composition:.1f}:  {describe(row[int(np.argmin(np.abs(result.compositions - composition)))])}")


def print_critical_temperatures(table):
    """A compute_critical_temperatures result as a table."""
    print(f"{'':34s} {'all x':>22} {'x < x0 - 0.05':>22}")
    for name, results in table.items():
        print(f"  {name:32s} " + " ".join(
            f"{'no gap':>22}" if math.isnan(temperature) else f"{'unstable at all T':>22}" if math.isinf(temperature)
            else f"{temperature:8.0f} K at x = {composition:.3f}" for temperature, composition in results))


def plot_critical_thickness(core_energies, core_radius, substrates=("AlN", "GaN"),
                            compositions=np.linspace(0.02, 0.5, 97)):
    """h_c(x) on the given substrates for several core-energy functions, with the chosen elastic data."""
    figure, axis = plt.subplots(figsize=(7.5, 4.5))
    for energy, style in zip(core_energies, ("-", "--", ":")):
        for substrate, colour in zip(substrates, ("tab:blue", "tab:green")):
            axis.plot(compositions, [critical_thickness_nm(x, energy, core_radius, substrate_in_plane_lattice[substrate])
                                     for x in compositions], style, color=colour,
                      label=f"on {substrate}, {energy.__name__.replace('dummy_core_energy_', '')}")
    axis.set_yscale("log"); axis.set_ylim(0.5, 1e3)
    axis.set_xlabel("Sc cation fraction $x$"); axis.set_ylabel("critical thickness $h_c$ (nm)")
    axis.legend(fontsize=7.5); axis.grid(alpha=0.25, which="both")
    print(f"Conditions: energy balance, elastic data '{chosen_elastic_data}'; core radius {core_radius.__name__}; "
          "all core values DUMMY")
    plt.tight_layout(); plt.show()



def plot_critical_thickness_by_elastic_data(core_energy, core_radius, compositions=np.linspace(0.02, 0.5, 97)):
    """h_c(x) on AlN and GaN for every elastic data option."""
    figure, axis = plt.subplots(figsize=(7.5, 4.5))
    for (name, constants), style in zip(elastic_data_options.items(), ("-", "--", ":")):
        for substrate, colour in (("AlN", "tab:blue"), ("GaN", "tab:green")):
            axis.plot(compositions, [critical_thickness_nm(x, core_energy, core_radius,
                                                           substrate_in_plane_lattice[substrate], constants)
                                     for x in compositions], style, color=colour, label=f"on {substrate}, {name}")
    axis.set_yscale("log"); axis.set_ylim(0.5, 1e3)
    axis.set_xlabel("Sc cation fraction $x$"); axis.set_ylabel("critical thickness $h_c$ (nm)")
    axis.legend(fontsize=7.5); axis.grid(alpha=0.25, which="both")
    print(f"Conditions: energy balance; core energy {core_energy.__name__}, core radius {core_radius.__name__} (DUMMY)")
    plt.tight_layout(); plt.show()



def print_interaction_parameter_shifts(substrates=("AlN", "GaN"), thicknesses_nm=(10.0, 100.0, 1000.0)):
    """Clamped strain energy as Redlich-Kister shifts, meV/f.u."""
    print(f"{'substrate':>9} {'h (nm)':>7} {'dL0':>8} {'dL1':>8} {'rel. rms':>9}   meV/f.u.; L0 = "
          f"{REDLICH_KISTER_WURTZITE[0] * MILLIELECTRONVOLT_PER_JOULE_PER_MOLE:.0f}")
    for substrate in substrates:
        for thickness_nm in thicknesses_nm:
            (delta_l0, delta_l1), relative = strain_as_interaction_parameters(
                substrate_in_plane_lattice[substrate], thickness_nm)
            print(f"{substrate:>9} {thickness_nm:7.0f} {delta_l0 * MILLIELECTRONVOLT_PER_JOULE_PER_MOLE:8.1f} "
                  f"{delta_l1 * MILLIELECTRONVOLT_PER_JOULE_PER_MOLE:8.1f} {relative:9.4f}")
    print(f"Conditions: Hirata film model, h_c with {chosen_core_energy.__name__}, {chosen_core_radius.__name__}")

# 6. Results

**How the result cells work.** Each cell computes into named variables, then plots them. The `if False or ... not in
globals()` guard skips the computation when the variables already exist, so replotting is instant. Change `False` to
`True` to recompute after changing an input. The variables hold plain arrays, for example `free_standing_coherent.binodal`
(temperature, composition) or `clamped_coherent.spinodal_curvature`.

**Another situation** is one `film_case(...)` call with different arguments. For example:

```python
film_case(coherent=True, substrate_lattice_parameter=substrate_in_plane_lattice["GaN"], film_thickness_nm=100.0,
          elastic_constants=elastic_data_options["zhang"])
```

A graded substrate passes a function of $x$ as `substrate_lattice_parameter`.

## 6.1 Critical thickness

In [ ]:
plot_critical_thickness((dummy_core_energy_high_constant, dummy_core_energy_low_constant, dummy_core_energy_high_slope),
                        dummy_core_radius)

## 6.2 Critical temperatures

In [ ]:
if False or "critical_temperatures_chosen" not in globals():         # True: recompute
    critical_temperatures_chosen = compute_critical_temperatures(standard_cases(chosen_elastic_constants))
print_critical_temperatures(critical_temperatures_chosen)

## 6.3 Elastic data compared

The critical thickness of 6.1 and the critical temperatures of 6.2 for every elastic data option of 2.2.3. The
free-standing incoherent case has no elastic energy and is the same for all of them.

In [ ]:
plot_critical_thickness_by_elastic_data(chosen_core_energy, chosen_core_radius)
if False or "critical_temperatures_by_elastic_data" not in globals():
    critical_temperatures_by_elastic_data = {
        name: compute_critical_temperatures(standard_cases(constants))
        for name, constants in elastic_data_options.items() if name != chosen_elastic_data}
for name, table in critical_temperatures_by_elastic_data.items():
    print(f"\nelastic data '{name}'")
    print_critical_temperatures(table)

## 6.4 Phase diagrams

Points: binodal; dotted: spinodal. Grey: the free-standing incoherent case, for reference.

### 6.4.1 Free-standing

In [ ]:
if False or "free_standing_coherent" not in globals():
    free_standing_incoherent = compute_phase_boundaries(film_case(coherent=False))
    free_standing_coherent = compute_phase_boundaries(film_case(coherent=True))
plot_phase_diagram(free_standing_incoherent, free_standing_coherent, title="Free-standing")

### 6.4.2 Fully clamped on AlN

The thinnest-film limit. Narrow bands near $x_0$ that stay unstable at every temperature come from the steep $a(x)$
at the transition under ~10 % compression; their strength depends on the assumed transition width. With hexagonal
elastic data the biaxial modulus dips at $x_0$ (to ~70 GPa for `"urban"`, ~20 GPa for `"zhang"`: $C_{33}$ extrapolated
to $x_0$), so domains at $x \approx x_0$ are cheap to clamp. The film then splits into AlN plus $x \approx x_0$ at every
temperature. This follows from the assumed construction of 2.2.3, not from data.

In [ ]:
if False or "clamped_coherent" not in globals():
    clamped_incoherent = compute_phase_boundaries(film_case(coherent=False, substrate_lattice_parameter=aluminium_nitride))
    clamped_coherent = compute_phase_boundaries(film_case(coherent=True, substrate_lattice_parameter=aluminium_nitride))
plot_phase_diagram(clamped_incoherent, clamped_coherent, reference=free_standing_incoherent,
                   title="Fully clamped on AlN")

### 6.4.3 1 µm film on AlN

Above $x \approx 0.43$ the critical thickness reaches the core radius, where the energy-balance model does not apply.

In [ ]:
if False or "micron_film_coherent" not in globals():
    micron_film_incoherent = compute_phase_boundaries(film_case(
        coherent=False, substrate_lattice_parameter=aluminium_nitride, film_thickness_nm=real_film_thickness_nm))
    micron_film_coherent = compute_phase_boundaries(film_case(
        coherent=True, substrate_lattice_parameter=aluminium_nitride, film_thickness_nm=real_film_thickness_nm))
plot_phase_diagram(micron_film_incoherent, micron_film_coherent, reference=free_standing_incoherent,
                   title="1 um film on AlN")

## 6.5 Decomposition driving force at growth temperatures

One line per growth temperature. The table lists the best two-domain state at `growth_temperature`, taken from the
same computed maps.

### 6.5.1 Free-standing

In [ ]:
if False or "free_standing_states" not in globals():
    free_standing_states = (
        compute_decomposition_map(film_case(coherent=False), growth_temperatures, driving_force_compositions),
        compute_decomposition_map(film_case(coherent=True), growth_temperatures, driving_force_compositions))
plot_driving_force(*free_standing_states, title="free-standing")
print_best_states(*free_standing_states, temperature=growth_temperature)

### 6.5.2 Fully clamped on AlN

In [ ]:
if False or "clamped_states" not in globals():
    clamped_states = tuple(compute_decomposition_map(
        film_case(coherent=coherent, substrate_lattice_parameter=aluminium_nitride),
        growth_temperatures, driving_force_compositions) for coherent in (False, True))
plot_driving_force(*clamped_states, title="fully clamped on AlN")
print_best_states(*clamped_states, temperature=growth_temperature)

### 6.5.3 1 µm film on AlN

In [ ]:
if False or "micron_film_states" not in globals():
    micron_film_states = tuple(compute_decomposition_map(
        film_case(coherent=coherent, substrate_lattice_parameter=aluminium_nitride,
                  film_thickness_nm=real_film_thickness_nm),
        growth_temperatures, driving_force_compositions) for coherent in (False, True))
plot_driving_force(*micron_film_states, title="1 um film on AlN")
print_best_states(*micron_film_states, temperature=growth_temperature)

## 6.6 Substrate strain as an interaction-parameter shift

If the relative rms is small, strain and $L^{(0)}$ are degenerate against a single measured boundary: separating them
needs boundaries at two or more thicknesses or substrates — a design requirement for Library 1.

In [ ]:
print_interaction_parameter_shifts()